In [ ]:
# 1/3 - clone + build debug_mean
import subprocess, os, shutil
d='/content/TokenVectorCuda'
subprocess.run('rm -rf '+d, shell=True)
subprocess.run('git clone --depth 1 --branch prototype/cuda-backend https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
ex=d+'/Release/cuda_backend/examples'
# write local .tkv (not yet committed) and compile with bundled compiler
tkv='''# -*- coding: utf-8 -*-
# @tkv.cuda_kernel grid="rows" block="256"
def debug_mean(x: "ptr[f32]", out: "ptr[f32]", rows: "i32", h: "i32") -> "void":
    s = shared_array(1024)
    r = bid()
    t = ltid()
    if r < rows:
        base = r * h
        if t < h:
            s[t] = x[base + t]
        syncthreads()
        stride = h / 2
        for k in range(10):
            if t < stride:
                s[t] = s[t] + s[t + stride]
            syncthreads()
            stride = stride / 2
        if t == 0:
            out[r] = s[0] / h


def run() -> "i32":
    return 0
'''
open(ex+'/debug_mean_local.tkv','w').write(tkv)
r=subprocess.run(d+'/Release/dist/tkvc-tv.exe '+ex+'/debug_mean_local.tkv '+ex+'/debug_mean_local.exe run', shell=True, capture_output=True, text=True)
print('compiler rc=',r.returncode, r.stdout[-300:], r.stderr[-300:])
print('cu exists',os.path.exists(ex+'/debug_mean_local.cuda.cu'))

In [ ]:
# 2/3 - build .so
import subprocess, shutil, os
ex='/content/TokenVectorCuda/Release/cuda_backend/examples'
cu=ex+'/debug_mean_local.cuda.cu'; so=ex+'/debug_mean_local.so'
nvcc=shutil.which('nvcc') or '/usr/local/cuda/bin/nvcc'
p=subprocess.run(nvcc+' -O2 -shared -std=c++20 --expt-relaxed-constexpr -x cu -Xcompiler -std=c++20 -Xcompiler -fPIC '+cu+' -o '+so+' -lcudart', shell=True, capture_output=True, text=True, timeout=600)
print('build rc=',p.returncode)
if p.returncode!=0: print(p.stderr[:2000])

In [ ]:
# 3/3 - dump mean vs torch
import ctypes, torch
ex='/content/TokenVectorCuda/Release/cuda_backend/examples'
lib=ctypes.CDLL(ex+'/debug_mean_local.so'); I32=ctypes.c_int; I64=ctypes.c_int64; F32=ctypes.c_float
fn=lib.tkv_launch_debug_mean; fn.argtypes=[I64,I64,I32,I32,I32,I32]; fn.restype=I32
h=256; rows=8
x=torch.randn(rows,h,device='cuda',dtype=torch.float32)
out=torch.zeros(rows,device='cuda',dtype=torch.float32)
rc=fn(x.data_ptr(),out.data_ptr(),rows,h,rows,256)
print('rc=',rc)
print('TKV mean:',out.cpu().numpy())
print('TORCH mean:',x.mean(dim=1).cpu().numpy())
print('maxerr=',(out-x.mean(dim=1)).abs().max().item())